# P3 single-date inference and T4 benchmark

Notebook này chỉ load checkpoint P3 T=1, benchmark và tạo map từ một ảnh.
Nó không mở nhãn training, split, optimizer hoặc checkpoint P2 T=6.

## 1. Mount Drive và checkout mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hongphuc-maker/xuan-thuy-prithvi-segmentation.git'
REPO_REF = 'main'  # Replace with the approved commit SHA before the scientific run.
REPO_ROOT = Path('/content/xuan-thuy-prithvi-segmentation')

if not (REPO_ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_ROOT)])
else:
    dirty = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('The Colab repository copy has local changes; use a fresh runtime or review them.')

subprocess.check_call(['git', '-C', str(REPO_ROOT), 'fetch', '--depth=1', 'origin', REPO_REF])
subprocess.check_call(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', 'FETCH_HEAD'])

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


## 2. Cài môi trường và restart sạch lần đầu

In [ ]:
import importlib.metadata
import os
import signal

ENV_MARKER = Path('/content/.prithvi_p3_single_env_v1')
if not ENV_MARKER.exists():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==2.3.5', 'terratorch==1.2.12', 'torchgeo==0.9.0', 'timm==1.0.30',
        'segmentation-models-pytorch==0.5.0', 'huggingface-hub==1.33.0',
        'einops==0.8.2', 'rasterio==1.5.2', 'pandas==2.2.3',
        'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'geopandas>=1,<2', 'tqdm>=4.66,<5',
        'git+https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git@0775770ae87b83b7d2982ed727226128685f3fba'
    ])
    ENV_MARKER.write_text('installed; restart required', encoding='utf-8')
    print('Environment installed. Colab will restart now; then run all cells from the top again.')
    os.kill(os.getpid(), signal.SIGKILL)

P3_SOURCE = REPO_ROOT / 'reference' / 'wce_completed'
sys.path.insert(0, str(P3_SOURCE))
sys.path.insert(0, str(REPO_ROOT))
print({name: importlib.metadata.version(name) for name in [
    'numpy', 'torch', 'terratorch', 'torchgeo', 'timm', 'rasterio', 'pandas'
]})
print('P3 source:', P3_SOURCE)


## 3. Khóa manifest, checkpoint P3 và output directory

In [ ]:
from pathlib import Path
import hashlib

DRIVE_ROOT = Path('/content/drive/MyDrive/Prithvi_Colab')
PROTOCOL_FOLDER = 'protocol_B'
DATASET_ID = 'thai_thuy_2026_sep21_single'
MANIFEST_PATH = (
    DRIVE_ROOT / 'data' / 'inference_single' / PROTOCOL_FOLDER
    / DATASET_ID / 'inference_dataset.json'
)
RUN_ROOT = DRIVE_ROOT / 'runs' / 'P3_single_sep21_UNet_WCE_REPRO'
if not (RUN_ROOT / 'TRAINING_COMPLETE.json').is_file():
    raise FileNotFoundError('P3 training is not complete: ' + str(RUN_ROOT))
CHECKPOINT_PATH = (
    RUN_ROOT / 'best_accepted.pt'
    if (RUN_ROOT / 'best_accepted.pt').is_file()
    else RUN_ROOT / 'best_any.pt'
)
BASE_CONFIG_PATH = (
    REPO_ROOT / 'configs' / 'experiments' / 'p3_prithvi_unet_wce_single.json'
)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert MANIFEST_PATH.is_file(), MANIFEST_PATH
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
assert BASE_CONFIG_PATH.is_file(), BASE_CONFIG_PATH
CHECKPOINT_HASH_PREFIX = sha256_file(CHECKPOINT_PATH)[:12]
OUTPUT_DIR = (
    DRIVE_ROOT / 'single_inference_runs' / PROTOCOL_FOLDER
    / DATASET_ID / CHECKPOINT_HASH_PREFIX
)
print('Manifest:', MANIFEST_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output:', OUTPUT_DIR)


## 4. Audit scene T=1 và strict-load checkpoint

In [ ]:
import torch
from IPython.display import display
from deployment.inference_pipeline import InferenceScene, load_wce_model

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU')
scene = InferenceScene(MANIFEST_PATH, verify=True, expected_frames=1)
model, inference_config, checkpoint_provenance, device = load_wce_model(
    BASE_CONFIG_PATH, scene.manifest, CHECKPOINT_PATH
)
assert len(scene.dates) == 1
assert inference_config['active_dates'] == scene.dates
display(scene.audit)
display(checkpoint_provenance)
print('CUDA device:', torch.cuda.get_device_name(device))


## 5. T4 benchmark

In [ ]:
import json
from deployment.inference_pipeline import benchmark_inference

device_name = torch.cuda.get_device_name(device)
if 'T4' not in device_name.upper():
    raise RuntimeError(f'This benchmark is reserved for T4; active device is {device_name}')
benchmark = benchmark_inference(model, scene, device, sample_tiles=30, warmup_tiles=5)
display(benchmark)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'T4_BENCHMARK.json').write_text(
    json.dumps(benchmark, indent=2), encoding='utf-8'
)


## 6. Tạo map single-date sau khi duyệt audit và benchmark

In [ ]:
from deployment.inference_pipeline import export_probability_map

RUN_MAP = False
if not RUN_MAP:
    print('Map inference is disabled. Review audit and benchmark, then set RUN_MAP=True.')
else:
    result = export_probability_map(
        model, scene, checkpoint_provenance, OUTPUT_DIR, device, progress=True
    )
    display(result)


## 7. Preview output đã hoàn tất

In [ ]:
import json
import matplotlib.pyplot as plt
import rasterio

marker = OUTPUT_DIR / 'INFERENCE_COMPLETE.json'
if not marker.exists():
    print('No completed map in this output directory.')
else:
    result = json.loads(marker.read_text())
    class_file = next(name for name in result['artifacts'] if name.startswith('classes_'))
    with rasterio.open(OUTPUT_DIR / class_file) as source:
        classes = source.read(1)
    plt.figure(figsize=(12, 8))
    plt.imshow(classes, vmin=0, vmax=11, cmap='tab20')
    plt.title(f"P3 T=1 | {result['protocol']} | {result['dataset_id']} | {result['target_date']}")
    plt.axis('off')
    plt.show()
    display(result)
